# Importer Walkthrough

This notebook walks through the local bulk-data import flow using the small `lightning_bolt` fixture bundle from `tests/fixtures/`.

The goal is to show how data lands in `mtg_cards`, gets enriched with external IDs, and then gains daily price snapshots.


In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root. Start Jupyter from somewhere inside the repo.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
FIXTURES_DIR = REPO_ROOT / "tests" / "fixtures"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


def reset_dir(path: Path) -> Path:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def copy_fixture_bundle(destination: Path, fixture_name: str, *filenames: str) -> dict[str, Path]:
    destination.mkdir(parents=True, exist_ok=True)
    copied: dict[str, Path] = {}
    for filename in filenames:
        source = FIXTURES_DIR / fixture_name / filename
        target = destination / filename
        shutil.copyfile(source, target)
        copied[filename] = target
    return copied

from mtg_source_stack.db.connection import connect
from mtg_source_stack.db.schema import initialize_database
from mtg_source_stack.importer.mtgjson import import_mtgjson_identifiers, import_mtgjson_prices
from mtg_source_stack.importer.scryfall import import_scryfall_cards

WORKDIR = reset_dir(REPO_ROOT / "var" / "walkthrough" / "02_importer")
DB_PATH = WORKDIR / "collection.db"
fixture_bundle = copy_fixture_bundle(
    WORKDIR,
    "lightning_bolt",
    "scryfall.json",
    "identifiers.json",
    "prices.json",
)
initialize_database(DB_PATH)
print(json.dumps({key: str(value) for key, value in fixture_bundle.items()}, indent=2))


## Step 1: Import Scryfall Printings

Scryfall defines the local printing rows that the rest of the runtime works from.


In [ ]:
scryfall_stats = import_scryfall_cards(DB_PATH, fixture_bundle["scryfall.json"])
print(scryfall_stats)

with connect(DB_PATH) as connection:
    catalog_rows = [dict(row) for row in connection.execute(
        "SELECT scryfall_id, name, set_code, collector_number, finishes_json FROM mtg_cards ORDER BY name"
    ).fetchall()]

print(json.dumps(catalog_rows, indent=2))


## Step 2: Import MTGJSON Identifiers

Identifier import enriches the existing `mtg_cards` rows with `mtgjson_uuid` and marketplace IDs.


In [ ]:
identifier_stats = import_mtgjson_identifiers(DB_PATH, fixture_bundle["identifiers.json"])
print(identifier_stats)

with connect(DB_PATH) as connection:
    enriched_rows = [dict(row) for row in connection.execute(
        "SELECT scryfall_id, mtgjson_uuid, tcgplayer_product_id FROM mtg_cards ORDER BY name"
    ).fetchall()]

print(json.dumps(enriched_rows, indent=2))


## Step 3: Import Daily Price Snapshots

Price import resolves MTGJSON UUIDs through the already-enriched local catalog and writes rows into `price_snapshots`.


In [ ]:
price_stats = import_mtgjson_prices(DB_PATH, fixture_bundle["prices.json"])
print(price_stats)

with connect(DB_PATH) as connection:
    price_rows = [dict(row) for row in connection.execute(
        "SELECT scryfall_id, provider, price_kind, finish, currency, snapshot_date, price_value FROM price_snapshots ORDER BY id"
    ).fetchall()]

print(json.dumps(price_rows, indent=2))


## End-To-End Check

At this point the local database is ready for inventory workflows: cards live in `mtg_cards`, external IDs are attached, and latest prices can be joined in from `price_snapshots`.


In [ ]:
with connect(DB_PATH) as connection:
    summary = {
        "mtg_cards": connection.execute("SELECT COUNT(*) FROM mtg_cards").fetchone()[0],
        "price_snapshots": connection.execute("SELECT COUNT(*) FROM price_snapshots").fetchone()[0],
        "inventories": connection.execute("SELECT COUNT(*) FROM inventories").fetchone()[0],
        "inventory_items": connection.execute("SELECT COUNT(*) FROM inventory_items").fetchone()[0],
    }

print(json.dumps(summary, indent=2))


## Next Step

Open `03_inventory_domain_walkthrough.ipynb` to move from imported reference data to actual personal inventory operations.
